# ALRA01 — AI Literature Research Assistant

의료/바이오 분야 국책과제 운영 연구원을 위한 문헌 조사 및 자료수집 서비스입니다.

이 노트북은 Day 8 가이드 형식을 따라 다음을 수행합니다.
- 프로젝트 폴더 생성
- `app/auth.py` 재사용
- `app/schemas.py`, `app/model_service.py`, `app/main_ALRA.py`, `frontend/app_ALRA.py` 작성
- FastAPI 서버 실행 및 Swagger 테스트
- Streamlit 프론트엔드 실행 안내

서비스 핵심 흐름은 `연구 주제 입력 -> PubMed 문헌 수집 -> 관련도 정렬 -> 연구 흐름 요약 -> research gap 도출 -> 1페이지 브리프 생성`입니다.


## 서비스 아이디어 요약

`AI Literature Research Assistant`는 연구 주제를 입력하면 관련 문헌을 자동 조사해 실무형 브리프로 정리하는 서비스입니다.

주요 대상 사용자
- 국책과제 운영 연구원
- 바이오/의료 PM
- 대학원 연구실
- 기술사업화 및 기업 R&D 전략기획 담당자

핵심 기능
- PubMed 기반 문헌 수집
- 실패 시 내장 샘플 코퍼스 폴백
- 관련도 기반 Top 논문 선별
- 연구 흐름 타임라인 생성
- research gap 도출
- 제안서에 활용 가능한 1페이지 문헌조사 브리프 생성


In [ ]:
import os

dirs = ["app", "models", "frontend"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("프로젝트 구조:")
print("""
my-project/
├── 📁 app/
│   ├── auth.py              ← Day 6에서 만든 것 그대로 재사용
│   ├── schemas.py           ← 입력/출력 스키마 정의
│   ├── sample_corpus.py     ← PubMed 실패 시 폴백용 샘플 코퍼스
│   ├── model_service.py     ← 모델 로드 + 문헌 조사 파이프라인
│   └── main_ALRA.py         ← FastAPI 서버
│
├── 📁 frontend/
│   └── app_ALRA.py          ← Streamlit UI
│
└── requirements.txt
""")


## 모델/파이프라인 사전 확인

이번 구현은 CPU 안정성과 과제 시연 재현성을 우선해, 무거운 생성 모델 대신 다음 전략을 사용합니다.
- 문헌 수집: PubMed E-utilities
- 관련도 계산: 경량 키워드 기반 점수화
- gap 도출: limitation/future-work 성격 문장 탐지
- 브리프 생성: 템플릿 기반 1페이지 초안

실서비스 확장 시 Hugging Face 임베딩/재랭커/요약 모델로 교체할 수 있도록 `model_service.py` 구조는 유지했습니다.


In [ ]:
import requests

query = "mRNA vaccine stability"
resp = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
    params={"db": "pubmed", "retmode": "json", "retmax": 3, "term": query},
    timeout=10,
)
print("상태 코드:", resp.status_code)
print(resp.json()["esearchresult"].get("idlist", []))


In [ ]:
%%writefile app/auth.py
"""
Day 6 - API Key 인증
"""
from fastapi import HTTPException, Header

# API Key 설정
# 실무에서는 환경 변수나 시크릿 매니저에서 로드합니다.
# 여기서는 학습 목적으로 하드코딩합니다.
VALID_API_KEYS = {
    "test-key-001": "사용자A",
    "test-key-002": "사용자B",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:  # *your code* — Header에서 키 추출
    """
    API Key를 검증합니다.
    FastAPI의 Depends()로 엔드포인트에 주입합니다.

    Returns:
        인증된 사용자 이름
    Raises:
        HTTPException: 키가 없거나 유효하지 않을 때
    """
    if x_api_key is None:
        raise HTTPException(
            status_code=401,                       # *your code* — Unauthorized
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )

    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )

    return VALID_API_KEYS[x_api_key]  # 사용자 이름 반환


In [ ]:
%%writefile app/schemas.py
"""
ALRA request/response schemas.
"""
from typing import Literal, Optional

from pydantic import BaseModel, Field, field_validator


ALLOWED_DOMAINS = {"bio/medical", "biomedical", "medicine", "pharma"}


class PredictRequest(BaseModel):
    query: str = Field(
        ...,
        min_length=5,
        max_length=300,
        description="문헌 조사를 수행할 연구 주제 또는 연구 질문",
        examples=["mRNA 백신 안정성 향상 기술"],
    )
    years: int = Field(
        default=5,
        ge=1,
        le=15,
        description="최근 몇 년의 문헌을 우선 조사할지 설정",
    )
    domain: str = Field(
        default="bio/medical",
        description="문헌 조사 도메인",
    )
    max_papers: int = Field(
        default=10,
        ge=5,
        le=50,
        description="최대 수집 논문 수",
    )
    language: Literal["english", "korean"] = Field(
        default="english",
        description="입력/출력 작업 언어",
    )

    @field_validator("query")
    @classmethod
    def validate_query(cls, value: str) -> str:
        cleaned = " ".join(value.split())
        if len(cleaned) < 5:
            raise ValueError("query는 최소 5자 이상이어야 합니다.")
        return cleaned

    @field_validator("domain")
    @classmethod
    def validate_domain(cls, value: str) -> str:
        normalized = value.strip().lower()
        if normalized not in ALLOWED_DOMAINS:
            raise ValueError(f"domain은 {sorted(ALLOWED_DOMAINS)} 중 하나여야 합니다.")
        return normalized


class PaperResult(BaseModel):
    title: str = Field(description="논문 제목")
    abstract_preview: str = Field(description="초록 미리보기")
    journal: Optional[str] = Field(default=None, description="저널명")
    year: Optional[int] = Field(default=None, description="출판 연도")
    doi: Optional[str] = Field(default=None, description="DOI")
    relevance_score: float = Field(description="관련도 점수")


class PredictResponse(BaseModel):
    success: bool = Field(description="성공 여부")
    query: str = Field(description="입력 연구 주제")
    source: Literal["pubmed", "fallback_sample"] = Field(description="문헌 수집 출처")
    papers: list[PaperResult] = Field(description="관련 논문 목록")
    timeline_summary: list[str] = Field(description="연구 흐름 요약")
    research_gaps: list[str] = Field(description="도출된 연구 공백")
    report_draft: str = Field(description="1페이지 문헌 조사 브리프")
    user: Optional[str] = Field(default=None, description="인증된 사용자")


In [ ]:
%%writefile app/sample_corpus.py
"""
ALRA fallback sample corpus.

PubMed 조회가 실패하거나 결과가 충분하지 않을 때 사용할
의료/바이오 분야 논문 메타데이터 샘플입니다.
"""

SAMPLE_PAPERS = [
    {
        "title": "Lipid nanoparticle engineering strategies for improving mRNA vaccine stability",
        "abstract": (
            "This review summarizes formulation strategies for improving the thermal "
            "stability of mRNA vaccines. It compares ionizable lipids, buffer systems, "
            "and lyophilization methods. Current evidence suggests that excipient "
            "selection and particle size control are major determinants of cold-chain "
            "robustness. However, long-term toxicity data and large-scale stability "
            "comparisons remain limited."
        ),
        "journal": "Advanced Drug Delivery Reviews",
        "year": 2024,
        "doi": "10.1016/j.addr.2024.01.001",
    },
    {
        "title": "Freeze-dried mRNA-LNP formulations for room-temperature distribution",
        "abstract": (
            "Researchers evaluated freeze-dried mRNA lipid nanoparticles for room-temperature "
            "storage. The study reports preserved antigen expression after reconstitution "
            "and highlights the potential for non-frozen logistics. Limitations include "
            "small animal sample sizes and limited real-world transport validation."
        ),
        "journal": "Nature Biomedical Engineering",
        "year": 2025,
        "doi": "10.1038/s41551-025-00021-7",
    },
    {
        "title": "Cold-chain optimization models for vaccine transportation in public health systems",
        "abstract": (
            "This paper presents optimization models for vaccine transportation and storage "
            "with a focus on minimizing spoilage risk and logistics cost. Results show "
            "that route redesign and dynamic temperature monitoring improve resilience. "
            "Future work should validate the model in low-resource settings."
        ),
        "journal": "Vaccine",
        "year": 2023,
        "doi": "10.1016/j.vaccine.2023.05.110",
    },
    {
        "title": "Comparative analysis of stabilizers in thermostable RNA vaccine platforms",
        "abstract": (
            "The authors compare sugar-based and polymer-based stabilizers across RNA vaccine "
            "platforms. Sucrose and trehalose improved short-term thermal resilience, while "
            "polymer systems showed better structural preservation. A key limitation is the "
            "lack of phase 3 clinical comparators."
        ),
        "journal": "Pharmaceutics",
        "year": 2022,
        "doi": "10.3390/pharmaceutics14091876",
    },
    {
        "title": "Translational challenges in room-temperature mRNA vaccine development",
        "abstract": (
            "This perspective reviews translational bottlenecks in room-temperature mRNA "
            "vaccines including regulatory uncertainty, manufacturing scale-up, and safety "
            "monitoring. The field is progressing rapidly, but consensus data on long-term "
            "storage toxicity and platform comparability are still missing."
        ),
        "journal": "Molecular Therapy",
        "year": 2024,
        "doi": "10.1016/j.ymthe.2024.03.014",
    },
    {
        "title": "AI-assisted prioritization of biomedical literature for national R&D planning",
        "abstract": (
            "The study proposes an AI-assisted workflow for biomedical literature triage, "
            "topic clustering, and policy reporting. The system improves review speed, but "
            "its ranking quality depends on metadata completeness and domain-specific prompts."
        ),
        "journal": "Journal of Biomedical Informatics",
        "year": 2023,
        "doi": "10.1016/j.jbi.2023.104321",
    },
    {
        "title": "Recent advances in thermostable vaccine delivery technologies",
        "abstract": (
            "A broad review of thermostable delivery approaches across protein, DNA, and RNA "
            "vaccines. The article highlights drying techniques, encapsulation materials, "
            "and packaging innovation. Evidence is promising, but standardized outcome "
            "measures are not yet widely adopted."
        ),
        "journal": "Trends in Biotechnology",
        "year": 2021,
        "doi": "10.1016/j.tibtech.2021.07.008",
    },
    {
        "title": "A bibliometric overview of mRNA vaccine stability research from 2020 to 2025",
        "abstract": (
            "Bibliometric analysis reveals rapid growth in mRNA vaccine stability research, "
            "especially after 2021. Dominant themes include lipid nanoparticle optimization, "
            "cold-chain reduction, and lyophilization. Underexplored areas include cost "
            "optimization, large-scale comparative trials, and toxicity monitoring."
        ),
        "journal": "Frontiers in Immunology",
        "year": 2025,
        "doi": "10.3389/fimmu.2025.123456",
    },
    {
        "title": "Clinical considerations for next-generation RNA vaccine storage",
        "abstract": (
            "Clinical deployment of next-generation RNA vaccines requires balancing "
            "immunogenicity, formulation stability, and supply-chain feasibility. Current "
            "clinical evidence remains fragmented across disease areas and formulation types."
        ),
        "journal": "The Lancet Digital Health",
        "year": 2024,
        "doi": "10.1016/S2589-7500(24)00042-6",
    },
    {
        "title": "Large language models for evidence synthesis in biomedical policy teams",
        "abstract": (
            "Large language models can accelerate evidence synthesis for biomedical policy "
            "teams by drafting summaries and identifying recurring themes. Risks include "
            "hallucinated citations and insufficient traceability. Hybrid review workflows "
            "with source grounding are recommended."
        ),
        "journal": "NPJ Digital Medicine",
        "year": 2024,
        "doi": "10.1038/s41746-024-01011-2",
    },
]


In [ ]:
%%writefile app/model_service.py
"""
ALRA model service.

외부 문헌 수집은 PubMed를 우선 사용하고,
실패하면 샘플 코퍼스로 자동 폴백합니다.
"""
from __future__ import annotations

import math
import re
from collections import Counter
from datetime import datetime
from typing import Any

import requests

from app.sample_corpus import SAMPLE_PAPERS

STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "how", "in",
    "into", "is", "it", "of", "on", "or", "that", "the", "this", "to", "using",
    "with", "기술", "연구", "분석", "개선", "향상", "대한", "및", "에서", "으로",
}

PUBMED_SEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUBMED_SUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"
PUBMED_ABSTRACT_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"


def load_model() -> dict[str, Any]:
    """
    Day 8 구조를 맞추기 위한 로더.
    실서비스에서는 HF 모델을 붙일 수 있도록 번들 구조를 유지한다.
    """
    return {
        "service_name": "AI Literature Research Assistant",
        "ranking_mode": "lightweight-keyword-ranking",
        "report_mode": "template-brief-generator",
    }


def _tokenize(text: str) -> list[str]:
    tokens = re.findall(r"[A-Za-z0-9가-힣\-]+", text.lower())
    return [token for token in tokens if token not in STOPWORDS and len(token) > 1]


def _normalize_whitespace(text: str) -> str:
    return " ".join((text or "").split())


def fetch_pubmed_papers(query: str, years: int, max_papers: int) -> list[dict[str, Any]]:
    current_year = datetime.utcnow().year
    min_year = max(current_year - years + 1, 2000)
    term = f"{query} AND ({min_year}:{current_year}[pdat])"

    search_resp = requests.get(
        PUBMED_SEARCH_URL,
        params={
            "db": "pubmed",
            "retmode": "json",
            "retmax": max_papers,
            "sort": "relevance",
            "term": term,
        },
        timeout=15,
    )
    search_resp.raise_for_status()
    id_list = search_resp.json()["esearchresult"].get("idlist", [])
    if not id_list:
        return []

    ids = ",".join(id_list)
    summary_resp = requests.get(
        PUBMED_SUMMARY_URL,
        params={"db": "pubmed", "retmode": "json", "id": ids},
        timeout=15,
    )
    summary_resp.raise_for_status()
    summary_data = summary_resp.json()["result"]

    abstract_resp = requests.get(
        PUBMED_ABSTRACT_URL,
        params={"db": "pubmed", "rettype": "abstract", "retmode": "text", "id": ids},
        timeout=20,
    )
    abstract_resp.raise_for_status()
    abstract_map = _parse_pubmed_abstracts(abstract_resp.text)

    papers: list[dict[str, Any]] = []
    for pmid in id_list:
        record = summary_data.get(pmid, {})
        title = _normalize_whitespace(record.get("title", ""))
        if not title:
            continue
        article_ids = record.get("articleids", [])
        doi = None
        for article_id in article_ids:
            if article_id.get("idtype") == "doi":
                doi = article_id.get("value")
                break

        pubdate = record.get("pubdate", "")
        year_match = re.search(r"(19|20)\d{2}", pubdate)
        year = int(year_match.group()) if year_match else None
        journal = record.get("fulljournalname") or record.get("source")
        abstract = abstract_map.get(pmid) or record.get("elocationid") or ""

        papers.append(
            {
                "title": title,
                "abstract": _normalize_whitespace(abstract),
                "journal": journal,
                "year": year,
                "doi": doi,
            }
        )

    return papers


def _parse_pubmed_abstracts(raw_text: str) -> dict[str, str]:
    blocks = re.split(r"\n\s*\n", raw_text)
    abstract_map: dict[str, str] = {}
    current_pmid = None

    for block in blocks:
        block = block.strip()
        if not block:
            continue
        pmid_match = re.search(r"PMID:\s*(\d+)", block)
        if pmid_match:
            current_pmid = pmid_match.group(1)
        if current_pmid is None:
            continue

        cleaned = re.sub(r"\s+", " ", block)
        cleaned = re.sub(r"PMID:\s*\d+.*$", "", cleaned).strip()
        if cleaned:
            abstract_map[current_pmid] = cleaned

    return abstract_map


def load_sample_corpus() -> list[dict[str, Any]]:
    return [paper.copy() for paper in SAMPLE_PAPERS]


def rank_papers(query: str, papers: list[dict[str, Any]]) -> list[dict[str, Any]]:
    query_tokens = _tokenize(query)
    if not query_tokens:
        return papers

    query_counter = Counter(query_tokens)
    query_norm = math.sqrt(sum(value * value for value in query_counter.values()))

    scored: list[dict[str, Any]] = []
    for paper in papers:
        text = f"{paper.get('title', '')} {paper.get('abstract', '')}"
        doc_tokens = _tokenize(text)
        if not doc_tokens:
            score = 0.0
        else:
            doc_counter = Counter(doc_tokens)
            overlap = sum(query_counter[token] * doc_counter.get(token, 0) for token in query_counter)
            doc_norm = math.sqrt(sum(value * value for value in doc_counter.values()))
            cosine = overlap / (query_norm * doc_norm) if doc_norm else 0.0

            title_bonus = 0.1 * sum(1 for token in query_tokens if token in _tokenize(paper.get("title", "")))
            recency_bonus = 0.02 * max((paper.get("year") or 2020) - 2020, 0)
            score = cosine + title_bonus + recency_bonus

        enriched = paper.copy()
        enriched["relevance_score"] = round(score, 4)
        scored.append(enriched)

    return sorted(scored, key=lambda item: item["relevance_score"], reverse=True)


def build_timeline_summary(top_papers: list[dict[str, Any]]) -> list[str]:
    if not top_papers:
        return ["관련 논문이 충분하지 않아 연구 흐름을 생성하지 못했습니다."]

    grouped: dict[int, list[str]] = {}
    for paper in top_papers:
        year = paper.get("year") or 0
        grouped.setdefault(year, []).append(f"{paper.get('title', '')} {paper.get('abstract', '')}")

    summary_lines: list[str] = []
    for year in sorted(grouped):
        joined = " ".join(grouped[year]).lower()
        if any(word in joined for word in ["freeze", "lyoph", "room-temperature", "ambient"]):
            theme = "동결건조 및 상온 유통 가능성 검토가 확대됨"
        elif any(word in joined for word in ["logistics", "cold-chain", "transport"]):
            theme = "콜드체인 최적화와 물류 비용 절감 연구가 두드러짐"
        elif any(word in joined for word in ["lipid", "lnp", "formulation", "stability"]):
            theme = "LNP 제형 안정화와 조성 최적화 중심 연구가 활발함"
        else:
            theme = "백신 안정성 및 실용화 관점의 후속 연구가 이어짐"
        label = str(year) if year else "연도 미상"
        summary_lines.append(f"{label}: {theme}")

    return summary_lines[:5]


def extract_research_gaps(top_papers: list[dict[str, Any]]) -> list[str]:
    sentences: list[str] = []
    for paper in top_papers:
        text = paper.get("abstract", "")
        for sentence in re.split(r"(?<=[.!?])\s+", text):
            lowered = sentence.lower()
            if any(
                keyword in lowered
                for keyword in ["limitation", "future work", "remain limited", "missing", "however", "lack"]
            ):
                sentences.append(sentence.strip())

    if not sentences:
        return [
            "장기 독성 및 장기 보관 안정성에 대한 후속 검증이 더 필요합니다.",
            "대규모 임상 또는 실제 물류 환경에서의 비교 연구가 부족합니다.",
            "비용 최적화와 규제 적용성을 함께 다룬 연구가 제한적입니다.",
        ]

    gap_buckets = {
        "독성·안전성 장기 데이터 부족": ["toxicity", "safety", "long-term"],
        "대규모 비교 임상 및 실증 검증 부족": ["phase 3", "large-scale", "clinical", "real-world"],
        "콜드체인 비용 및 운영 최적화 연구 부족": ["cost", "logistics", "transport", "cold-chain"],
        "표준화된 평가 지표와 플랫폼 비교 부족": ["standard", "comparator", "comparability", "outcome"],
    }

    found: list[str] = []
    lowered_blob = " ".join(sentences).lower()
    for gap, keywords in gap_buckets.items():
        if any(keyword in lowered_blob for keyword in keywords):
            found.append(gap)

    if not found:
        found = [
            "제형 안정성과 실제 배포 조건을 함께 검증한 통합 연구가 부족합니다.",
            "임상 적용성과 공급망 운영성을 동시에 비교한 연구가 제한적입니다.",
        ]

    return found[:4]


def generate_report_draft(
    query: str,
    top_papers: list[dict[str, Any]],
    timeline_summary: list[str],
    research_gaps: list[str],
    source: str,
) -> str:
    lead_titles = [paper["title"] for paper in top_papers[:3]]
    lead_journals = [paper.get("journal") for paper in top_papers[:3] if paper.get("journal")]
    journal_text = ", ".join(lead_journals[:3]) if lead_journals else "주요 생의학 저널"

    return (
        f"[문헌조사 브리프]\n"
        f"주제: {query}\n"
        f"문헌 출처: {'PubMed 실시간 수집' if source == 'pubmed' else '내장 샘플 코퍼스 폴백'}\n\n"
        f"1. 배경\n"
        f"- 본 주제는 의료/바이오 R&D 기획 및 선행연구 정리에 직접 연결되는 이슈입니다.\n"
        f"- 수집된 문헌은 {journal_text}를 중심으로 정리되었습니다.\n\n"
        f"2. 핵심 논문\n"
        + "\n".join(f"- {title}" for title in lead_titles)
        + "\n\n3. 연구 흐름\n"
        + "\n".join(f"- {line}" for line in timeline_summary)
        + "\n\n4. 연구 공백\n"
        + "\n".join(f"- {gap}" for gap in research_gaps)
        + "\n\n5. 제안 시사점\n"
        f"- 후속 과제 기획 시 제형 안정성, 임상 검증, 공급망 운영성의 세 축을 함께 평가하는 연구 프레임이 유효합니다.\n"
        f"- 특히 상온 유통 가능성, 대규모 비교 검증, 비용 최적화 지표를 결합한 과제가 차별화 포인트가 될 수 있습니다.\n"
    )


def predict(model_bundle: dict[str, Any], input_data: dict[str, Any]) -> dict[str, Any]:
    query = input_data["query"]
    years = input_data["years"]
    max_papers = input_data["max_papers"]

    try:
        papers = fetch_pubmed_papers(query=query, years=years, max_papers=max_papers)
        source = "pubmed" if papers else "fallback_sample"
    except Exception:
        papers = []
        source = "fallback_sample"

    if not papers:
        papers = load_sample_corpus()

    ranked = rank_papers(query, papers)[:max_papers]
    timeline_summary = build_timeline_summary(ranked[: min(5, len(ranked))])
    research_gaps = extract_research_gaps(ranked[: min(8, len(ranked))])
    report_draft = generate_report_draft(
        query=query,
        top_papers=ranked,
        timeline_summary=timeline_summary,
        research_gaps=research_gaps,
        source=source,
    )

    paper_results = [
        {
            "title": paper["title"],
            "abstract_preview": (paper.get("abstract", "")[:280] + "...") if len(paper.get("abstract", "")) > 280 else paper.get("abstract", ""),
            "journal": paper.get("journal"),
            "year": paper.get("year"),
            "doi": paper.get("doi"),
            "relevance_score": paper.get("relevance_score", 0.0),
        }
        for paper in ranked
    ]

    return {
        "success": True,
        "query": query,
        "source": source,
        "papers": paper_results,
        "timeline_summary": timeline_summary,
        "research_gaps": research_gaps,
        "report_draft": report_draft,
    }


In [ ]:
%%writefile app/main_ALRA.py
"""
ALRA FastAPI server.
"""
import asyncio
from concurrent.futures import ThreadPoolExecutor

from fastapi import Depends, FastAPI, HTTPException

from app.auth import verify_api_key
from app.error_handlers import register_error_handlers
from app.logger_config import setup_logger
from app.middleware import RequestLoggingMiddleware
from app.model_service import load_model, predict
from app.schemas import PredictRequest, PredictResponse

logger = setup_logger("alra_api")

app = FastAPI(
    title="AI Literature Research Assistant API",
    description="의료/바이오 문헌 조사 브리프 생성 API (인증 필요)",
    version="1.0.0",
)

app.add_middleware(RequestLoggingMiddleware)
register_error_handlers(app)

inference_executor = ThreadPoolExecutor(max_workers=4, thread_name_prefix="alra")
model_bundle = None


@app.on_event("startup")
async def startup() -> None:
    global model_bundle
    logger.info("ALRA 서비스 로드 중...")
    model_bundle = load_model()
    logger.info("ALRA 서비스 로드 완료")


def run_predict(request_dict: dict) -> dict:
    if model_bundle is None:
        raise RuntimeError("서비스가 아직 로드되지 않았습니다.")
    return predict(model_bundle, request_dict)


@app.get("/health", tags=["System"])
async def health_check() -> dict:
    return {
        "status": "healthy" if model_bundle is not None else "loading",
        "model": model_bundle["service_name"] if model_bundle else None,
        "ranking_mode": model_bundle["ranking_mode"] if model_bundle else None,
    }


@app.post("/predict", response_model=PredictResponse, tags=["Prediction"])
async def predict_literature(
    request: PredictRequest,
    user: str = Depends(verify_api_key),
):
    if model_bundle is None:
        raise HTTPException(status_code=503, detail="서비스가 아직 로드되지 않았습니다.")

    logger.info("문헌 조사 요청 - 사용자: %s, query: %s", user, request.query)

    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            inference_executor,
            run_predict,
            request.model_dump(),
        )
    except Exception as exc:
        logger.exception("문헌 조사 실패")
        raise HTTPException(status_code=500, detail=f"문헌 조사 실패: {exc}") from exc

    return PredictResponse(**result, user=user)


In [ ]:
%%writefile frontend/app_ALRA.py
"""
ALRA Streamlit frontend.
"""
import requests
import streamlit as st

API_BASE = "http://localhost:8000"

st.set_page_config(
    page_title="AI Literature Research Assistant",
    page_icon="📚",
    layout="wide",
)


def call_api(payload: dict, api_key: str):
    try:
        response = requests.post(
            f"{API_BASE}/predict",
            json=payload,
            headers={"X-API-Key": api_key},
            timeout=60,
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.ConnectionError:
        st.error("🔌 서버에 연결할 수 없습니다. FastAPI 서버를 먼저 실행하세요.")
        return None
    except requests.exceptions.HTTPError as exc:
        if exc.response.status_code == 401:
            st.error("🔑 인증 실패입니다. API Key를 확인하세요.")
        elif exc.response.status_code == 422:
            try:
                detail = exc.response.json().get("detail")
                st.error(f"입력 검증 오류: {detail}")
            except Exception:
                st.error("입력 검증 오류가 발생했습니다.")
        else:
            st.error(f"서버 오류가 발생했습니다. (HTTP {exc.response.status_code})")
        return None
    except Exception as exc:
        st.error(f"알 수 없는 오류가 발생했습니다: {type(exc).__name__}")
        return None


def fetch_health():
    try:
        response = requests.get(f"{API_BASE}/health", timeout=5)
        response.raise_for_status()
        return response.json()
    except Exception:
        return None


with st.sidebar:
    st.header("설정")
    api_key = st.text_input("API Key", value="test-key-001", type="password")
    years = st.slider("최근 조사 연도", min_value=1, max_value=15, value=5)
    max_papers = st.slider("수집 논문 수", min_value=5, max_value=30, value=10)
    domain = st.selectbox("분야", ["bio/medical", "biomedical", "medicine", "pharma"], index=0)
    language = st.selectbox("언어", ["english", "korean"], index=0)
    st.divider()

    health = fetch_health()
    if health and health.get("status") == "healthy":
        st.success("🟢 서버 연결됨")
        st.caption(f"서비스: {health.get('model', 'N/A')}")
        st.caption(f"랭킹: {health.get('ranking_mode', 'N/A')}")
    elif health:
        st.warning("🟡 서비스 로딩 중...")
    else:
        st.error("🔴 서버 연결 실패")

    st.divider()
    st.caption("AI Literature Research Assistant")


st.title("AI Literature Research Assistant")
st.write("의료/바이오 분야 연구 주제를 입력하면 관련 문헌 조사 브리프를 생성합니다.")

query = st.text_area(
    "연구 주제 또는 연구 질문",
    value="mRNA 백신 안정성 향상 기술",
    height=120,
    help="예: mRNA 백신 안정성 향상 기술, 동결건조 기반 상온 유통 백신 플랫폼",
)

if st.button("문헌 조사 시작", type="primary", use_container_width=True):
    payload = {
        "query": query,
        "years": years,
        "domain": domain,
        "max_papers": max_papers,
        "language": language,
    }
    with st.spinner("문헌을 수집하고 브리프를 생성하는 중입니다..."):
        result = call_api(payload, api_key)
    if result:
        st.session_state["alra_result"] = result


result = st.session_state.get("alra_result")
if result:
    col1, col2 = st.columns([1.3, 1.0])

    with col1:
        st.subheader("Top 논문 리스트")
        st.caption(f"문헌 출처: {result['source']}")
        for index, paper in enumerate(result["papers"], start=1):
            with st.expander(f"{index}. {paper['title']}"):
                st.write(f"**관련도 점수:** {paper['relevance_score']}")
                st.write(f"**저널:** {paper.get('journal') or 'N/A'}")
                st.write(f"**연도:** {paper.get('year') or 'N/A'}")
                doi = paper.get("doi") or "N/A"
                st.write(f"**DOI:** {doi}")
                st.write(f"**초록 미리보기:** {paper['abstract_preview'] or '초록 정보 없음'}")

    with col2:
        st.subheader("연구 흐름 Timeline")
        for line in result["timeline_summary"]:
            st.write(f"- {line}")

        st.subheader("Research Gap")
        for gap in result["research_gaps"]:
            st.write(f"- {gap}")

    st.subheader("1페이지 문헌조사 브리프")
    st.text_area("보고서 초안", value=result["report_draft"], height=320)


## 서버 실행

아래 셀은 FastAPI 서버를 백그라운드 스레드로 실행합니다.
Swagger UI는 `http://localhost:8000/docs` 에서 확인할 수 있습니다.


In [ ]:
import nest_asyncio
import threading
import time
import uvicorn

nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main_ALRA:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨 — http://localhost:8000/docs")


## API 테스트

평가 기준에 맞춰 health check, 정상 추론, 인증 실패, 입력 검증 실패를 확인합니다.


In [ ]:
import requests
import json

API_URL = "http://localhost:8000"
HEADERS = {"X-API-Key": "test-key-001"}

print("[health]")
print(requests.get(f"{API_URL}/health").json())

payload = {
    "query": "mRNA 백신 안정성 향상 기술",
    "years": 5,
    "domain": "bio/medical",
    "max_papers": 10,
    "language": "english",
}

response = requests.post(f"{API_URL}/predict", json=payload, headers=HEADERS, timeout=60)
print("[predict] 상태:", response.status_code)
print(json.dumps(response.json(), ensure_ascii=False, indent=2)[:2000])


In [ ]:
import requests

API_URL = "http://localhost:8000"

resp_no_key = requests.post(
    f"{API_URL}/predict",
    json={
        "query": "mRNA 백신 안정성 향상 기술",
        "years": 5,
        "domain": "bio/medical",
        "max_papers": 10,
        "language": "english",
    },
    timeout=30,
)
print("API Key 없음:", resp_no_key.status_code, resp_no_key.json())

resp_bad = requests.post(
    f"{API_URL}/predict",
    json={
        "query": "abc",
        "years": 50,
        "domain": "bio/medical",
        "max_papers": 100,
        "language": "english",
    },
    headers={"X-API-Key": "test-key-001"},
    timeout=30,
)
print("잘못된 입력:", resp_bad.status_code)
print(resp_bad.json())


## Streamlit 실행 안내

터미널 또는 노트북 셀에서 아래 명령으로 프론트엔드를 실행합니다.

```bash
streamlit run frontend/app_ALRA.py
```

입력 항목
- API Key
- 최근 조사 연도
- 수집 논문 수
- 분야
- 언어
- 연구 주제

결과 화면
- Top 논문 리스트
- relevance score
- 초록 미리보기
- DOI
- 연구 흐름 timeline
- research gap
- 1페이지 문헌조사 브리프


## Day 8 최종 체크포인트

Q1. 본 프로젝트에서 Pydantic 검증은 어떤 잘못된 입력을 막아줍니까?  
A1. 빈 질의, 지나치게 짧거나 긴 질의, 허용 범위를 벗어난 기간과 논문 수, 허용되지 않은 도메인 값을 차단합니다.

Q2. `Depends(verify_api_key)`를 제거하면 어떤 위험이 있습니까?  
A2. 누구나 API를 호출할 수 있어 무단 사용, 과도한 트래픽, 리소스 낭비, 연구 데이터 처리 남용 위험이 생깁니다.

Q3. `run_in_executor`를 사용한 이유는 무엇입니까?  
A3. PubMed 조회와 문헌 정리 로직이 동기적으로 동작해도 이벤트 루프를 막지 않도록 별도 스레드에서 추론을 처리하기 위해서입니다.

Q4. Day 1~8 중 가장 많이 참고한 Day는 어디였습니까? 왜?  
A4. Day 5와 Day 7입니다. FastAPI 서버 구조, Pydantic 응답 모델, Streamlit API 호출 패턴을 그대로 확장했기 때문입니다.

Q5. 이 서비스를 실제로 배포하려면 추가로 무엇이 필요합니까?  
A5. Docker 패키징, 환경 변수 기반 인증키 관리, 로깅/모니터링, 외부 API 장애 대응, 검색 인덱스 고도화, 배포 인프라 구성이 필요합니다.


## 프로젝트 소개 및 회고

### <프로젝트 소개>
본 프로젝트는 의료/바이오 분야 연구자를 위한 `AI Literature Research Assistant` 서비스입니다. 연구 주제를 입력하면 PubMed에서 관련 문헌을 수집하고, 관련도 기반으로 핵심 논문을 정리한 뒤 연구 흐름과 연구 공백을 요약하여 브리프를 생성합니다. 의료 연구과제 기획이나 선행연구 조사 업무를 빠르게 지원하는 것이 목표입니다.

### <구현 과정>
Day 8 가이드를 기준으로 FastAPI 백엔드와 Streamlit 프론트엔드를 분리해 구성했습니다. 인증은 Day 6의 `auth.py`를 그대로 재사용했고, `/predict` 엔드포인트에서 Pydantic 검증과 `run_in_executor` 비동기 추론 구조를 유지했습니다. 문헌 수집은 PubMed 실연동을 우선 적용했고, 네트워크 문제나 결과 부재 상황을 대비해 샘플 코퍼스 폴백(데모 버전인 것을 감안하여 문제없이 서버와 UI가 작동하는 것에 초점을 둠)을 넣었습니다.

### <결과 요약>
- Swagger UI에서 `/predict` 테스트 가능
- API Key 없이 요청하면 401 반환
- 잘못된 입력에는 422 반환
- Streamlit UI에서 연구 주제 입력 후 결과 확인 가능

### <회고>
이번 프로젝트에서는 기존 학습 내용을 조합해 하나의 서비스 형태로 묶는 경험이 가장 중요했습니다. 특히 FastAPI 구조 재사용, 입력 검증, 인증, 프론트엔드 연동이 실제 서비스 뼈대를 만든다는 점을 확인할 수 있었습니다. 이후에는 임베딩 기반 검색, 재랭킹 모델, PDF 다운로드, 다중 데이터 소스 확장을 적용하면 더 완성도 높은 서비스로 발전시킬 수 있을 것 같습니다.
